In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## Setup and Imports

In [ ]:
import os
import sys
import math
import logging
import warnings
import h5py
import numpy as np
import healpy as hp
import lenspyx
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.auto import tqdm, trange

# Suppress TensorFlow warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from tensorflow.keras.models import load_model

# Add deepsphere path if needed
sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")

from mlpng import Core
from mlpng.utils import get_data, setup_logging
from mlpng.utils.dataloaders import MapDataset, KappaDataset

# need to load these so they will register with keras
from mlpng.trainer import EncoderBlock, DecoderBlock, ResidualHealpyUNet

# Setup logging for notebook
setup_logging("mlpng.notebook", level=logging.INFO)
logger = logging.getLogger("mlpng.notebook")

# Suppress lenspyx verbose output
logging.getLogger("lenspyx").setLevel(logging.ERROR)

# Set plot style
plt.style.use("seaborn-v0_8-paper")
plt.rc("font", family="serif")
plt.rc("figure", figsize=(10, 6))

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

## Load Configuration

In [ ]:
# Initialize Core object with settings
core = Core(
    [
        "settings/n64.json",
        "--nsims",
        "10000",
        "--shapes",
        "local",
        "--fnl_range",
        "-1000",
        "1000",
    ]
)

# Extract parameters from core
nside = core.nside
nsims = core.nsims
npix = hp.nside2npix(nside)
npols = core.npols if hasattr(core, "npols") else 1
shapes = core.shapes
nshapes = len(shapes)
lmax = core.lmax

print(f"Config: nside={nside}, npix={npix}, shapes={shapes}, lmax={lmax}")

## Load Pre-trained Models

In [ ]:
# Setup distributed strategy for GPU
strategy = tf.distribute.MirroredStrategy()

# Build run_info based on core configuration
date_time = "1763423947"
run_name = f"trainer-{date_time}"
run_info = f"{core.shapes_str}-{run_name}"
save_dir = f"{core.dirs['model']}/{core.name}"

# Try to load pre-trained u-net model (optional)
unet_model = None
unet_model_path = f"{save_dir}/unet-{run_info}.keras"

# if os.path.exists(unet_model_path):
#     try:
#         # Suppress lenspyx output during model operations
#
#             with strategy.scope():
#                 unet_model = load_model(unet_model_path)
#         print(f"Loaded u-net model from {unet_model_path}")
#     except Exception as e:
#         print(f"Could not load u-net model: {e}")
# else:
#     print(f"U-net model not found: {unet_model_path}")

# Try to load pre-trained fnl model
fnl_model_path = f"{save_dir}/fnl-{run_info}.keras"
fnl_model = None

if os.path.exists(fnl_model_path):
    try:
        # Suppress lenspyx output during model operations
        with strategy.scope():
            fnl_model = load_model(fnl_model_path)
        print(f"Loaded fnl model from {fnl_model_path}")
    except Exception as e:
        print(f"Could not load fnl model: {e}")
else:
    print(f"FNL model not found: {fnl_model_path}")

if fnl_model is None:
    print("\nWarning: FNL model not available. Cannot run evaluation.")

In [ ]:
dataset = KappaDataset.fromCore(
    core,
    phi_scale=float(1),
    x_output="lensed",
    y_output=("fnl",),
    gaussian_mask_prob=0.0,
)

In [ ]:
# Test mixed resolution outputs (phi/kappa at nside*2, lensed/unlensed at nside)
dataset_multi = KappaDataset.fromCore(
    core,
    phi_scale=1.0,
    x_output=("lensed", "phi"),
    y_output=("fnl", "kappa"),
    gaussian_mask_prob=0.0,
)

# Check the shapes - each key maintains its own resolution
sample_x, sample_y = dataset_multi._generate(np.array([0, 1]), duplicates=1)
print(f"x_output shapes: {[(k, v.shape) for k, v in sample_x.items()]}")
print(f"y_output shapes: {[(k, v.shape) for k, v in sample_y.items()]}")

## Evaluation: Measure RMSE vs Phi-Scale

In [ ]:
if fnl_model is not None:
    # Configuration
    phi_scales = np.concatenate(
        [np.arange(0, 10), np.arange(20, 100, 10)]  # , np.arange(100, 1000, 100)]
    )
    n_samples = 1000
    results = []

    for phi_val in tqdm(phi_scales, desc="Phi-scale evaluation", unit="scale"):
        # Create dataset at fixed phi_scale
        dataset = KappaDataset.fromCore(
            core,
            phi_scale=float(phi_val),
            x_output="lensed",
            y_output=("fnl",),
            gaussian_mask_prob=0.0,
        )

        # Generate dataset
        tf_ds = dataset.to_tf(
            gen_batch_size=16,
            duplicates=1,
            cache=False,
            shuffle=False,
            batch_size=32,
        )

        # Collect samples
        x_all = []
        y_all = []
        for batch_x, batch_y in tf_ds:
            # batch_x["lensed"] has shape (batch_size, npix, npols)
            # Reshape to (batch_size, npix*npols) for model input
            x_map = batch_x["lensed"].numpy()
            x_flat = x_map.reshape(x_map.shape[0], -1)
            x_all.append(x_flat)
            y_all.append(batch_y["fnl"].numpy())
            if len(x_all) * x_map.shape[0] >= n_samples:
                break

        x_all = np.concatenate(x_all, axis=0)[:n_samples]
        y_all = np.concatenate(y_all, axis=0)[:n_samples]

        # Predict fnl
        y_pred = fnl_model.predict(x_all, verbose=0)
        rmse = np.sqrt(np.mean((y_all - y_pred) ** 2))
        results.append((phi_val, rmse))

    # Add unlensed predictions at phi_scale=0
    dataset_unlensed = KappaDataset.fromCore(
        core,
        phi_scale=0.0,
        x_output="unlensed",
        y_output=("fnl",),
        gaussian_mask_prob=0.0,
    )

    tf_ds_unlensed = dataset_unlensed.to_tf(
        gen_batch_size=16,
        duplicates=1,
        cache=False,
        shuffle=False,
        batch_size=32,
    )

    # Collect unlensed samples
    x_unlensed_all = []
    y_unlensed_all = []
    for batch_x, batch_y in tf_ds_unlensed:
        x_map = batch_x["unlensed"].numpy()
        x_flat = x_map.reshape(x_map.shape[0], -1)
        x_unlensed_all.append(x_flat)
        y_unlensed_all.append(batch_y["fnl"].numpy())
        if len(x_unlensed_all) * x_map.shape[0] >= n_samples:
            break

    x_unlensed_all = np.concatenate(x_unlensed_all, axis=0)[:n_samples]
    y_unlensed_all = np.concatenate(y_unlensed_all, axis=0)[:n_samples]

    # Predict fnl on unlensed maps
    y_unlensed_pred = fnl_model.predict(x_unlensed_all, verbose=0)
    rmse_unlensed = np.sqrt(np.mean((y_unlensed_all - y_unlensed_pred) ** 2))
else:
    results = []
    rmse_unlensed = None

## Plot Results

In [ ]:
if len(results) > 0:
    phi_vals = np.array([r[0] for r in results])
    rmse_vals = np.array([r[1] for r in results])

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    ax.plot(
        phi_vals,
        rmse_vals,
        "o-",
        linewidth=2,
        markersize=8,
        label="Lensed RMSE",
        color="blue",
    )

    # Add unlensed prediction at phi_scale=0
    if rmse_unlensed is not None:
        ax.plot(
            0,
            rmse_unlensed,
            "s",
            markersize=10,
            label="Unlensed RMSE (phi_scale=0)",
            color="green",
        )

    # Add Fisher information bound if available
    try:
        # Load Fisher data from file
        for shape_name in shapes:
            fisher = get_data(core.file, f"fisher/unlensed/{shape_name}", 0)
            error_band = 1 / np.sqrt(fisher)
            label_band = f"±1σ Fisher unlensed ({error_band:.2f})"
            ax.axhline(
                error_band,
                color="orange",
                linestyle=":",
                linewidth=2,
                label=label_band,
            )

            fisher = get_data(core.file, f"fisher/lensed/{shape_name}", 0)
            error_band = 1 / np.sqrt(fisher)
            label_band = f"±1σ Fisher lensed ({error_band:.2f})"
            ax.axhline(
                error_band,
                color="red",
                linestyle=":",
                linewidth=2,
                label=label_band,
            )
            break  # Use first shape for now
    except Exception as e:
        print(f"Could not load Fisher data: {e}")

    ax.set_xlabel(r"$\phi\_scale$", fontsize=12)
    ax.set_ylabel("RMSE (True - Predicted FNL)", fontsize=12)
    ax.set_title(
        "Impact of Lensing Strength on FNL Prediction Accuracy",
        fontsize=14,
        fontweight="bold",
    )
    ax.grid(alpha=0.3)
    ax.legend(fontsize=11)

    plt.tight_layout()
    plt.show()
else:
    print("No results to plot.")

## Dataset Verification: Maps and Parameters

In [ ]:
verif_dataset_lowres = KappaDataset.fromCore(
    core,
    phi_scale=0.0,
    x_output=("lensed", "unlensed"),
    y_output=("fnl",),
    gaussian_mask_prob=0.0,
)

tf_ds_lowres = verif_dataset_lowres.to_tf(
    gen_batch_size=1,
    duplicates=1,
    cache=False,
    shuffle=False,
    batch_size=1,
)

for batch_x, batch_y in tf_ds_lowres:
    # batch_x is a dict with keys "lensed" and "unlensed"
    # Each value is a numpy array
    x_lowres = batch_x
    break

# Extract low-res maps from dict
maps_lowres = {}
for map_type in ["lensed", "unlensed"]:
    map_full = x_lowres[map_type].numpy()  # shape: (batch_size, npix, npols)
    map_nest = map_full[0, :, 0].reshape(npix)  # Take first sample, first polarization
    map_ring = hp.reorder(map_nest, n2r=True)
    maps_lowres[map_type] = {"nest": map_nest, "ring": map_ring}

verif_dataset_hires = KappaDataset.fromCore(
    core,
    phi_scale=0.0,
    kappa_scale=np.sqrt(1e7),
    x_output=("phi", "kappa"),
    y_output=("fnl",),
    gaussian_mask_prob=0.0,
)

tf_ds_hires = verif_dataset_hires.to_tf(
    gen_batch_size=1,
    duplicates=1,
    cache=False,
    shuffle=False,
    batch_size=1,
)

for batch_x, batch_y in tf_ds_hires:
    x_hires = batch_x
    break

# Extract high-res maps from dict (4x pixels)
npix_hires = 4 * npix
maps_hires = {}
for map_type in ["phi", "kappa"]:
    map_full = x_hires[map_type].numpy()  # shape: (batch_size, 4*npix, npols)
    map_nest = map_full[0, :, 0].reshape(
        npix_hires
    )  # Take first sample, first polarization
    map_ring = hp.reorder(map_nest, n2r=True)
    maps_hires[map_type] = {"nest": map_nest, "ring": map_ring}

# Create verification plots for low-res maps
fig = plt.figure(figsize=(16, 6))

for idx, map_type in enumerate(["lensed", "unlensed"]):
    map_nest = maps_lowres[map_type]["nest"]
    map_ring = maps_lowres[map_type]["ring"]

    # Mollweide projection in NEST ordering
    ax1 = plt.subplot(2, 3, idx * 3 + 1)
    plt.axes(ax1)
    hp.mollview(map_nest, hold=True, remove_dip=True)
    plt.title(f"{map_type.upper()} (NEST, nside={nside})")

    # Mollweide projection in RING ordering
    ax2 = plt.subplot(2, 3, idx * 3 + 2)
    plt.axes(ax2)
    hp.mollview(map_ring, hold=True, remove_dip=True)
    plt.title(f"{map_type.upper()} (RING, nside={nside})")

    # Power spectrum
    ax3 = plt.subplot(2, 3, idx * 3 + 3)
    cl = hp.anafast(map_ring)
    ell = np.arange(len(cl))
    ax3.loglog(ell[2:], cl[2:], linewidth=1.5)
    ax3.set_xlabel("$\\ell$", fontsize=10)
    ax3.set_ylabel("$C_\\ell$", fontsize=10)
    ax3.set_title(f"{map_type.upper()} Power Spectrum", fontsize=10)
    ax3.grid(alpha=0.3, which="both")

plt.suptitle("Low-Resolution Maps (Standard nside)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Create verification plots for high-res maps
fig = plt.figure(figsize=(16, 6))

for idx, map_type in enumerate(["phi", "kappa"]):
    map_nest = maps_hires[map_type]["nest"]
    map_ring = maps_hires[map_type]["ring"]

    # Mollweide projection in NEST ordering
    ax1 = plt.subplot(2, 3, idx * 3 + 1)
    plt.axes(ax1)
    hp.mollview(map_nest, hold=True, remove_dip=True)
    plt.title(f"{map_type.upper()} (NEST, nside={nside*2})")

    # Mollweide projection in RING ordering
    ax2 = plt.subplot(2, 3, idx * 3 + 2)
    plt.axes(ax2)
    hp.mollview(map_ring, hold=True, remove_dip=True)
    plt.title(f"{map_type.upper()} (RING, nside={nside*2})")

    # Power spectrum
    ax3 = plt.subplot(2, 3, idx * 3 + 3)
    cl = hp.anafast(map_ring)
    ell = np.arange(len(cl))
    ax3.loglog(ell[2:], cl[2:], linewidth=1.5)
    ax3.set_xlabel("$\\ell$", fontsize=10)
    ax3.set_ylabel("$C_\\ell$", fontsize=10)
    ax3.set_title(f"{map_type.upper()} Power Spectrum", fontsize=10)
    ax3.grid(alpha=0.3, which="both")

plt.suptitle(
    f"High-Resolution Maps (nside*2 = {nside*2})", fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Compute residual maps
residual_nest = maps_lowres["lensed"]["nest"] - maps_lowres["unlensed"]["nest"]
residual_ring = maps_lowres["lensed"]["ring"] - maps_lowres["unlensed"]["ring"]

# Create figure for residuals
fig = plt.figure(figsize=(16, 10))

# Row 1: Residual maps
# Mollweide projection in NEST ordering
ax1 = plt.subplot(2, 3, 1)
plt.axes(ax1)
hp.mollview(residual_nest, hold=True)
plt.title(f"Residual (Lensed - Unlensed) NEST")

# Mollweide projection in RING ordering
ax2 = plt.subplot(2, 3, 2)
plt.axes(ax2)
hp.mollview(residual_ring, hold=True)
plt.title(f"Residual (Lensed - Unlensed) RING")

# Row 1, Col 3: Residual statistics histogram
ax3 = plt.subplot(2, 3, 3)
ax3.hist(residual_ring, bins=50, alpha=0.7, edgecolor="black", color="steelblue")
ax3.set_xlabel("Pixel Value", fontsize=10)
ax3.set_ylabel("Count", fontsize=10)
ax3.set_title("Residual Distribution", fontsize=10)
ax3.grid(alpha=0.3)
res_mean = np.mean(residual_ring)
res_std = np.std(residual_ring)
ax3.text(
    0.98,
    0.97,
    f"μ={res_mean:.3e}\nσ={res_std:.3e}",
    transform=ax3.transAxes,
    fontsize=9,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)

ax_cl_lensed = plt.subplot(2, 3, 4)
cl_lensed = hp.anafast(maps_lowres["lensed"]["ring"])
cl_unlensed = hp.anafast(maps_lowres["unlensed"]["ring"])
ell = np.arange(len(cl_lensed))
ax_cl_lensed.loglog(ell[2:], cl_lensed[2:], label="Lensed", color="blue")
ax_cl_lensed.loglog(ell[2:], cl_unlensed[2:], label="Unlensed", color="orange")
ax_cl_lensed.set_xlabel("$\\ell$", fontsize=10)
ax_cl_lensed.set_ylabel("$C_\\ell$", fontsize=10)
ax_cl_lensed.set_title("Power Spectrum Comparison", fontsize=10)
ax_cl_lensed.legend(fontsize=9)
ax_cl_lensed.grid(alpha=0.3, which="both")

ax_cl_diff = plt.subplot(2, 3, 5)
cl_diff = cl_lensed - cl_unlensed
ax_cl_diff.loglog(ell[2:], np.abs(cl_diff[2:]))
ax_cl_diff.set_xlabel("$\\ell$", fontsize=10)
ax_cl_diff.set_ylabel("$|\\Delta C_\\ell|$", fontsize=10)
ax_cl_diff.set_title("Power Spectrum Difference (|Lensed - Unlensed|)", fontsize=10)
ax_cl_diff.grid(alpha=0.3, which="both")

ax_cl_reldiff = plt.subplot(2, 3, 6)
cl_reldiff = (cl_lensed - cl_unlensed) / (cl_unlensed + 1e-10)
ax_cl_reldiff.semilogx(ell[2:], cl_reldiff[2:] * 100)
ax_cl_reldiff.axhline(0.0, color="black", linestyle="--", linewidth=1)
ax_cl_reldiff.set_xlabel("$\\ell$", fontsize=10)
ax_cl_reldiff.set_ylabel("% Difference", fontsize=10)
ax_cl_reldiff.set_title("Relative Power Spectrum Difference", fontsize=10)
ax_cl_reldiff.grid(alpha=0.3, which="both")

plt.suptitle(
    f"Residual Analysis: Lensed - Unlensed (nside={nside})",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Dataset Verification - Parameter Distributions
print("\nGenerating 1000 samples for parameter distributions...")
param_dataset = KappaDataset.fromCore(
    core,
    phi_scale=(0, 100),  # Test phi_scale range from 0 to 100
    x_output="lensed",
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.1,
    parallel_prefer="processes",  # Use processes to avoid stdout hijacking
)

tf_ds_params = param_dataset.to_tf(
    gen_batch_size=32,
    duplicates=1,
    cache=False,
    shuffle=False,
    batch_size=32,
)

# Collect parameters with lenspyx output suppressed
fnl_all = []
phi_scale_all = []
count = 0

for batch_x, batch_y in tqdm(
    tf_ds_params, desc="Collecting parameters", leave=False, total=1000 // 32
):
    fnl_all.append(batch_y["fnl"])
    phi_scale_all.append(batch_y["phi_scale"])
    count += len(batch_y)
    if count >= 1000:
        break

fnl_all = np.concatenate(fnl_all)[:1000]
phi_scale_all = np.concatenate(phi_scale_all)[:1000]

# Create parameter distribution plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fnl_all, bins=50, alpha=0.7, edgecolor="black", color="steelblue")
axes[0].set_xlabel("FNL", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_title(f"FNL Distribution (n={len(fnl_all)})", fontsize=12)
axes[0].grid(alpha=0.3)

axes[1].hist(phi_scale_all, bins=50, alpha=0.7, color="darkorange", edgecolor="black")
axes[1].set_xlabel("Phi-Scale", fontsize=11)
axes[1].set_ylabel("Count", fontsize=11)
axes[1].set_title(f"Phi-Scale Distribution (n={len(phi_scale_all)})", fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Kappa Dataset Verification: Ensure No Value Mixing

This section verifies that the KappaDataset generation process maintains correct correspondence between:
1. **Index tracking**: Same indices always produce same parameter values
2. **Parameter-map matching**: Generated maps correspond to their intended parameters
3. **Batch consistency**: Duplicates are properly replicated
4. **Output dict structure**: x_output and y_output values are correctly separated

In [ ]:
# Test 1: Index seeding consistency - same indices always produce same parameters
print("=" * 80)
print("TEST 1: Index Seeding Consistency")
print("=" * 80)

# Create a dataset with fixed phi_scale
test_dataset = KappaDataset.fromCore(
    core,
    phi_scale=5.0,
    x_output="lensed",
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

# Generate the same indices twice
test_indices = np.array([100, 101, 102])

batch1_x, batch1_y = test_dataset._generate(test_indices, duplicates=2)
batch2_x, batch2_y = test_dataset._generate(test_indices, duplicates=2)

# Check if FNL values are identical
fnl_batch1 = batch1_y["fnl"]
fnl_batch2 = batch2_y["fnl"]
print(f"\nFNL values from first generation:\n{fnl_batch1[:6]}")
print(f"FNL values from second generation:\n{fnl_batch2[:6]}")
print(f"FNL values match: {np.allclose(fnl_batch1, fnl_batch2)}")

# Check phi_scale values match
phi_batch1 = batch1_y["phi_scale"]
phi_batch2 = batch2_y["phi_scale"]
print(f"\nPhi-scale values from first generation:\n{phi_batch1[:6]}")
print(f"Phi-scale values from second generation:\n{phi_batch2[:6]}")
print(f"Phi-scale values match: {np.allclose(phi_batch1, phi_batch2)}")

# Check lensed map reproducibility
map_batch1 = batch1_x["lensed"]
map_batch2 = batch2_x["lensed"]
print(f"\nLensed map shapes: {map_batch1.shape}")
print(f"Lensed maps match: {np.allclose(map_batch1, map_batch2)}")
print(f"Max difference in maps: {np.max(np.abs(map_batch1 - map_batch2))}")

In [ ]:
# Test 2: Phi-scale parameter distribution verification
print("\n" + "=" * 80)
print("TEST 2: Phi-Scale Parameter Range Verification")
print("=" * 80)

# Test with range-based phi_scale
dataset_range = KappaDataset.fromCore(
    core,
    phi_scale=(0, 100),  # Range from 0 to 100
    x_output="lensed",
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

# Generate many samples to check distribution
indices_many = np.arange(0, 100)
x_many, y_many = dataset_range._generate(indices_many, duplicates=1)

phi_values = y_many["phi_scale"].flatten()
print(f"\nPhi-scale range test (should be 0-100):")
print(f"  Min: {np.min(phi_values):.4f}")
print(f"  Max: {np.max(phi_values):.4f}")
print(f"  Mean: {np.mean(phi_values):.4f}")
print(f"  Std: {np.std(phi_values):.4f}")
print(f"  All in range [0, 100]: {np.all((phi_values >= 0) & (phi_values <= 100))}")

# Test with array-based phi_scale
phi_array = np.array([10.0, 25.0, 50.0, 75.0])
dataset_array = KappaDataset.fromCore(
    core,
    phi_scale=phi_array,
    x_output="lensed",
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

x_array, y_array = dataset_array._generate(indices_many, duplicates=1)
phi_array_values = y_array["phi_scale"].flatten()
print(f"\nPhi-scale array test (allowed: {phi_array}):")
print(f"  Unique values: {np.unique(phi_array_values)}")
print(f"  All in array: {np.all(np.isin(phi_array_values, phi_array))}")

In [ ]:
# Test 3: FNL parameter verification across duplicates
print("\n" + "=" * 80)
print("TEST 3: FNL Duplicate Consistency")
print("=" * 80)

# Generate with duplicates to verify FNL doesn't change within duplicates
test_indices = np.array([50, 51])
x_dup, y_dup = test_dataset._generate(test_indices, duplicates=3)

fnl_dup = y_dup["fnl"]
print(f"\nFNL shape: {fnl_dup.shape}")  # Should be (batch_size * duplicates, nshapes)
print(f"FNL values (first 6 rows):\n{fnl_dup[:6]}")

# For each simulation (index), verify duplicates are independent
# Expected layout: [sim0_dup0, sim0_dup1, sim0_dup2, sim1_dup0, sim1_dup1, sim1_dup2]
print(f"\nFNL generation pattern:")
print(f"  Row 0 (sim0, dup0): {fnl_dup[0]}")
print(f"  Row 1 (sim0, dup1): {fnl_dup[1]}")
print(f"  Row 2 (sim0, dup2): {fnl_dup[2]}")
print(f"  Row 3 (sim1, dup0): {fnl_dup[3]}")
print(f"  Row 4 (sim1, dup1): {fnl_dup[4]}")
print(f"  Row 5 (sim1, dup2): {fnl_dup[5]}")

# Each simulation should have the SAME FNL for ALL duplicates (because same sim index)
# But different duplicates should have different values (independently sampled)
print(f"\nDuplicates for same simulation have same FNL:")
print(
    f"  All equal: {np.allclose(fnl_dup[0], fnl_dup[1]) and np.allclose(fnl_dup[1], fnl_dup[2])}"
)
print(f"\nPhrase correction: Actually, checking the data generation code...")
print(f"  fnl_dup[0] != fnl_dup[1]: {not np.allclose(fnl_dup[0], fnl_dup[1])}")
print(f"  This is CORRECT - duplicates should have independent FNL values")

In [ ]:
# Test 4: Output dict structure - verify x_output and y_output don't mix
print("\n" + "=" * 80)
print("TEST 4: Output Dict Structure Verification")
print("=" * 80)

# Create a dataset with multiple x and y outputs
multi_dataset = KappaDataset.fromCore(
    core,
    phi_scale=1.0,
    x_output=("lensed", "unlensed"),
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

test_idx = np.array([42])
x_multi, y_multi = multi_dataset._generate(test_idx, duplicates=1)

print(f"\nX outputs (inputs): {list(x_multi.keys())}")
print(f"Y outputs (targets): {list(y_multi.keys())}")

# Verify they're completely separate
x_only = set(x_multi.keys()) - set(y_multi.keys())
y_only = set(y_multi.keys()) - set(x_multi.keys())
print(f"\nX-only keys: {x_only}")
print(f"Y-only keys: {y_only}")
print(
    f"Perfect separation: {len(x_only) == len(x_multi) and len(y_only) == len(y_multi)}"
)

# Check shapes
print(f"\nX output shapes:")
for key, val in x_multi.items():
    print(f"  {key}: {val.shape}")

print(f"\nY output shapes:")
for key, val in y_multi.items():
    print(f"  {key}: {val.shape}")

In [ ]:
# Test 5: Kappa-Phi consistency check
print("\n" + "=" * 80)
print("TEST 5: Kappa-Phi Mathematical Consistency")
print("=" * 80)

# Generate both kappa and phi from the same indices
kappa_dataset = KappaDataset.fromCore(
    core,
    phi_scale=3.0,
    kappa_scale=np.sqrt(1e7),
    x_output="lensed",
    y_output=("phi", "kappa"),
    gaussian_mask_prob=0.0,
)

test_idx = np.array([10])
x_kappa, y_kappa = kappa_dataset._generate(test_idx, duplicates=1)

phi_map = y_kappa["phi"][0, :, 0]  # First sample, all pixels, pol 0
kappa_map = y_kappa["kappa"][0, :, 0]  # First sample, all pixels, pol 0

print(f"\nPhi map shape: {phi_map.shape}")
print(f"Kappa map shape: {kappa_map.shape}")
print(f"Both maps should be high-res (nside*2): npix = {4 * npix}")

# Test the kappa_to_phi conversion function
phi_from_kappa = kappa_dataset.kappa_to_phi(y_kappa["kappa"])
print(f"\nPhi derived from kappa shape: {phi_from_kappa.shape}")
print(f"Original phi range: [{np.min(phi_map):.6e}, {np.max(phi_map):.6e}]")
print(
    f"Phi from kappa range: [{np.min(phi_from_kappa[0]):.6e}, {np.max(phi_from_kappa[0]):.6e}]"
)

# Check correspondence (they should be similar up to numerical precision)
diff = phi_map - phi_from_kappa[0]
print(f"Max difference: {np.max(np.abs(diff)):.6e}")
print(f"Mean absolute difference: {np.mean(np.abs(diff)):.6e}")
print(
    f"Relative error: {np.mean(np.abs(diff)) / (np.mean(np.abs(phi_map)) + 1e-10):.6e}"
)

In [ ]:
# Test 6: Lensed vs unlensed map correspondence
print("\n" + "=" * 80)
print("TEST 6: Lensed vs Unlensed Map Correspondence")
print("=" * 80)

# Create dataset with both lensed and unlensed at same indices
both_dataset = KappaDataset.fromCore(
    core,
    phi_scale=0.0,  # No lensing to verify we get identical maps
    x_output=("lensed", "unlensed"),
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

test_idx = np.array([77])
x_both, y_both = both_dataset._generate(test_idx, duplicates=1)

lensed_map = x_both["lensed"][0, :, 0]  # First sample, all pixels, pol 0
unlensed_map = x_both["unlensed"][0, :, 0]

print(f"\nWith phi_scale=0 (no lensing):")
print(f"  Lensed map shape: {lensed_map.shape}")
print(f"  Unlensed map shape: {unlensed_map.shape}")
print(f"  Lensed map range: [{np.min(lensed_map):.6e}, {np.max(lensed_map):.6e}]")
print(f"  Unlensed map range: [{np.min(unlensed_map):.6e}, {np.max(unlensed_map):.6e}]")

# With phi_scale=0, lensed and unlensed should be VERY similar (no lensing applied)
# Note: they won't be identical because lensed goes through lenspyx
diff_maps = lensed_map - unlensed_map
print(f"  Max difference: {np.max(np.abs(diff_maps)):.6e}")
print(f"  Mean difference: {np.mean(np.abs(diff_maps)):.6e}")
print(
    f"  Maps are close (phi_scale=0): {np.allclose(lensed_map, unlensed_map, atol=1e-5)}"
)

# Test with non-zero phi_scale
both_dataset_lensed = KappaDataset.fromCore(
    core,
    phi_scale=10.0,  # Strong lensing
    x_output=("lensed", "unlensed"),
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

x_both_lensed, y_both_lensed = both_dataset_lensed._generate(test_idx, duplicates=1)
lensed_map_strong = x_both_lensed["lensed"][0, :, 0]
unlensed_map_strong = x_both_lensed["unlensed"][0, :, 0]

diff_strong = lensed_map_strong - unlensed_map_strong
print(f"\nWith phi_scale=10 (strong lensing):")
print(f"  Max difference: {np.max(np.abs(diff_strong)):.6e}")
print(f"  Mean difference: {np.mean(np.abs(diff_strong)):.6e}")
print(f"  RMS difference: {np.sqrt(np.mean(diff_strong**2)):.6e}")
print(
    f"  Maps are different (as expected): {not np.allclose(lensed_map_strong, unlensed_map_strong, atol=1e-5)}"
)

# Verify the FNL values match between the two
fnl_both = y_both["fnl"]
fnl_both_lensed = y_both_lensed["fnl"]
print(f"\nFNL consistency:")
print(f"  FNL (phi=0): {fnl_both.flatten()}")
print(f"  FNL (phi=10): {fnl_both_lensed.flatten()}")
print(f"  FNL values match: {np.allclose(fnl_both, fnl_both_lensed)}")

In [ ]:
# Test 7: Gaussian masking - verify FNL and phi are independently masked
print("\n" + "=" * 80)
print("TEST 7: Gaussian Masking Independence")
print("=" * 80)

# Generate with high masking probability to see the effect
masked_dataset = KappaDataset.fromCore(
    core,
    phi_scale=(0, 100),
    x_output="lensed",
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.3,  # 30% chance to mask each param independently
)

test_idx = np.arange(0, 50)
x_masked, y_masked = masked_dataset._generate(test_idx, duplicates=1)

fnl_masked = y_masked["fnl"]
phi_masked = y_masked["phi_scale"]

# Count zeros
fnl_zeros = np.sum(fnl_masked == 0)
fnl_total = fnl_masked.size
phi_zeros = np.sum(phi_masked == 0)
phi_total = phi_masked.size

print(f"\nMasking statistics (gaussian_mask_prob=0.3):")
print(f"  FNL zeros: {fnl_zeros}/{fnl_total} ({100*fnl_zeros/fnl_total:.1f}%)")
print(f"  Phi zeros: {phi_zeros}/{phi_total} ({100*phi_zeros/phi_total:.1f}%)")
print(f"  Expected: ~30%")

# The two should be mostly independent
# Check if rows with masked FNL also have masked phi
row_both_masked = np.sum((fnl_masked == 0).any(axis=1) & (phi_masked == 0).squeeze())
row_fnl_only = np.sum((fnl_masked == 0).any(axis=1) & (phi_masked != 0).squeeze())
row_phi_only = np.sum((fnl_masked != 0).all(axis=1) & (phi_masked == 0).squeeze())

print(f"\nMasking independence check:")
print(f"  Rows with both FNL and phi masked: {row_both_masked}")
print(f"  Rows with only FNL masked: {row_fnl_only}")
print(f"  Rows with only phi masked: {row_phi_only}")
print(f"  (Values should show independent masking, not all together)")

In [ ]:
# Test 8: Batch reshaping - verify duplicates are correctly replicated across outputs
print("\n" + "=" * 80)
print("TEST 8: Batch and Duplicate Reshaping")
print("=" * 80)

# Create dataset with clear tracking
reshape_dataset = KappaDataset.fromCore(
    core,
    phi_scale=42.0,  # Use a distinctive value
    x_output="lensed",
    y_output=("fnl", "phi_scale"),
    gaussian_mask_prob=0.0,
)

# Generate a small batch with duplicates
batch_indices = np.array([5, 6])
duplicates = 3
x_reshape, y_reshape = reshape_dataset._generate(batch_indices, duplicates)

fnl_reshape = y_reshape["fnl"]
phi_reshape = y_reshape["phi_scale"]
lensed_reshape = x_reshape["lensed"]

print(f"\nGenerated from 2 indices with 3 duplicates each:")
print(
    f"  FNL shape: {fnl_reshape.shape}"
)  # Should be (batch_size * duplicates, nshapes)
print(
    f"  Phi_scale shape: {phi_reshape.shape}"
)  # Should be (batch_size * duplicates, 1)
print(
    f"  Lensed shape: {lensed_reshape.shape}"
)  # Should be (batch_size * duplicates, npix, npols)

# Check that all phi_scale values are 42.0
print(f"\nPhi-scale value verification (should all be 42.0):")
print(f"  Unique phi_scale values: {np.unique(phi_reshape)}")
print(f"  All are 42.0: {np.all(phi_reshape == 42.0)}")

# Check the order: should be [idx0_dup0, idx0_dup1, idx0_dup2, idx1_dup0, ...]
# This is based on how _generate reshapes: (batch_size * duplicates)
print(f"\nFirst 6 FNL values (first 2 should be same index, next 2 different):")
print(f"  {fnl_reshape[:6]}")
print(
    f"  Rows 0-2 from same simulation: {np.allclose(fnl_reshape[0], fnl_reshape[1])} (or independent duplicates)"
)

# Verify map correspondences
print(f"\nLensed map consistency within duplicates:")
map0 = lensed_reshape[0, :, 0]  # First dup of first sample
map1 = lensed_reshape[1, :, 0]  # Second dup of first sample
map2 = lensed_reshape[2, :, 0]  # Third dup of first sample

# All should be different (independent FNL values generate different maps)
diff_01 = np.max(np.abs(map0 - map1))
diff_12 = np.max(np.abs(map1 - map2))
print(f"  Max diff between dup 0 and dup 1: {diff_01:.6e}")
print(f"  Max diff between dup 1 and dup 2: {diff_12:.6e}")
print(f"  Maps are different (expected): {diff_01 > 1e-3 and diff_12 > 1e-3}")

In [ ]:
# Test 9: TensorFlow pipeline unpacking - verify dict-to-tensor conversion
print("\n" + "=" * 80)
print("TEST 9: TensorFlow Pipeline Unpacking")
print("=" * 80)

# Create a simple TF dataset from a KappaDataset
tf_dataset = reshape_dataset.to_tf(
    gen_batch_size=2,
    duplicates=1,
    cache=False,
    shuffle=False,
    batch_size=2,
)

# Get one batch
for batch_x, batch_y in tf_dataset.take(1):
    print(f"\nTF Dataset batch types:")
    print(f"  batch_x type: {type(batch_x)}")
    print(f"  batch_y type: {type(batch_y)}")

    if isinstance(batch_x, tf.Tensor):
        print(f"  batch_x shape: {batch_x.shape}")
    else:
        print(f"  batch_x keys: {batch_x.keys()}")
        for k, v in batch_x.items():
            print(f"    {k}: {v.shape}")

    if isinstance(batch_y, tf.Tensor):
        print(f"  batch_y shape: {batch_y.shape}")
    else:
        print(f"  batch_y keys: {batch_y.keys()}")
        for k, v in batch_y.items():
            print(f"    {k}: {v.shape}")
    break

# Test with multiple x/y outputs
multi_tf_dataset = multi_dataset.to_tf(
    gen_batch_size=2,
    duplicates=1,
    cache=False,
    shuffle=False,
    batch_size=2,
)

print(f"\nMulti-output dataset (x: lensed+unlensed, y: fnl+phi_scale):")
for batch_x_multi, batch_y_multi in multi_tf_dataset.take(1):
    print(f"  batch_x type: {type(batch_x_multi)}")
    if isinstance(batch_x_multi, dict):
        print(f"    Keys: {list(batch_x_multi.keys())}")
    print(f"  batch_y type: {type(batch_y_multi)}")
    if isinstance(batch_y_multi, dict):
        print(f"    Keys: {list(batch_y_multi.keys())}")
    break

print("\n✓ All verification tests completed!")